# Customer Churn Intelligence

## Objective

Inspect the Telco Customer Churn dataset to understand schema, data types, missing values, target distribution, and initial data-quality issues.

**Stage:** Data Understanding (Step 3) — no modeling, splitting, or preprocessing yet.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

In [ ]:
DATA_PATH = Path("../data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv")

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Dataset not found at {DATA_PATH.resolve()}. "
        "Download WA_Fn-UseC_-Telco-Customer-Churn.csv from Kaggle and place it in data/raw/."
    )

df = pd.read_csv(DATA_PATH)
print(f"Loaded: {DATA_PATH}")

## 1. First Look

In [ ]:
df.head()

In [ ]:
print("Shape:", df.shape)
print("\nColumn names:")
print(df.columns.tolist())

In [ ]:
df.dtypes

In [ ]:
df.info()

In [ ]:
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = df.select_dtypes(include=["object"]).columns.tolist()

print("Numeric columns:", numeric_cols)
print("\nDescriptive statistics (numeric):")
display(df[numeric_cols].describe())

In [ ]:
print("Categorical columns:", categorical_cols)
print("\nDescriptive statistics (categorical — counts of top categories):")
display(df[categorical_cols].describe(include="object"))

In [ ]:
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)

missing_df = pd.DataFrame({
    "Missing Values": missing,
    "Missing %": missing_pct,
})
missing_df[missing_df["Missing Values"] > 0]

In [ ]:
print("Duplicated rows:", df.duplicated().sum())

In [ ]:
unique_counts = df.nunique().sort_values()
unique_counts

In [ ]:
target_col = "Churn"

churn_counts = df[target_col].value_counts()
churn_pct = (df[target_col].value_counts(normalize=True) * 100).round(2)

target_summary = pd.DataFrame({
    "Count": churn_counts,
    "Percentage": churn_pct,
})
target_summary

## 2. Column Role Identification

In [ ]:
identifier_cols = ["customerID"]
target_col = "Churn"

numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
object_cols = df.select_dtypes(include=["object"]).columns.tolist()

# Exclude identifier and target from feature typing
categorical_cols = [c for c in object_cols if c not in identifier_cols + [target_col]]

binary_categorical_cols = [
    col
    for col in categorical_cols + numeric_cols
    if col not in identifier_cols + [target_col] and df[col].nunique() == 2
]

multi_class_categorical_cols = [
    col for col in categorical_cols if df[col].nunique() > 2
]

print("Numeric columns:", numeric_cols)
print("Categorical columns (object, excl. ID/target):", categorical_cols)
print("Binary columns (2 unique values):", binary_categorical_cols)
print("Multi-class categorical columns:", multi_class_categorical_cols)
print("Identifier columns:", identifier_cols)
print("Target column:", target_col)

In [ ]:
# Flag columns that may need cleaning or special handling during preprocessing
total_charges_coerced = pd.to_numeric(df["TotalCharges"], errors="coerce")
total_charges_non_numeric = total_charges_coerced.isna().sum()

suspicious_cols = []
if df["TotalCharges"].dtype == "object":
    suspicious_cols.append("TotalCharges (stored as object, not float)")
if total_charges_non_numeric > 0:
    suspicious_cols.append(
        f"TotalCharges ({total_charges_non_numeric} blank/non-numeric values)"
    )
if df["customerID"].nunique() != len(df):
    suspicious_cols.append("customerID (not unique across rows)")
if df["SeniorCitizen"].dtype != "object":
    suspicious_cols.append("SeniorCitizen (encoded as int64 0/1, not Yes/No strings)")

internet_related = [
    "OnlineSecurity", "OnlineBackup", "DeviceProtection",
    "TechSupport", "StreamingTV", "StreamingMovies",
]
for col in internet_related:
    if "No internet service" in df[col].unique():
        suspicious_cols.append(f"{col} (contains 'No internet service' sentinel category)")

if "No phone service" in df["MultipleLines"].unique():
    suspicious_cols.append("MultipleLines (contains 'No phone service' sentinel category)")

print("Suspicious columns / notes:")
for note in suspicious_cols:
    print(f"  - {note}")

## 3. Focused Inspection

In [ ]:
print("customerID — sample values:")
display(df["customerID"].head(10))
print(f"Unique customerID: {df['customerID'].nunique()} / {len(df)} rows")
print(f"Duplicate customerID: {df['customerID'].duplicated().sum()}")

In [ ]:
print("Churn — value counts:")
display(df["Churn"].value_counts())
print("\nChurn — percentages:")
display((df["Churn"].value_counts(normalize=True) * 100).round(2))

In [ ]:
print("TotalCharges — dtype:", df["TotalCharges"].dtype)
print("Sample values:")
display(df["TotalCharges"].head(10))

blank_mask = df["TotalCharges"].astype(str).str.strip().eq("")
print(f"\nBlank/whitespace-only TotalCharges rows: {blank_mask.sum()}")

total_charges_numeric = pd.to_numeric(df["TotalCharges"], errors="coerce")
non_numeric_mask = total_charges_numeric.isna()
print(f"Non-numeric after coercion: {non_numeric_mask.sum()}")

if non_numeric_mask.any():
    print("\nRows with non-numeric TotalCharges:")
    display(
        df.loc[non_numeric_mask, ["customerID", "tenure", "MonthlyCharges", "TotalCharges", "Churn"]]
    )

In [ ]:
print("tenure — summary:")
display(df["tenure"].describe())
print(f"\nCustomers with tenure = 0: {(df['tenure'] == 0).sum()}")

In [ ]:
print("MonthlyCharges — summary:")
display(df["MonthlyCharges"].describe())

## 4. Data Dictionary (Initial)

In [ ]:
def initial_role_note(col: str) -> str:
    """Assign an initial role/note based on observed column properties."""
    if col == "customerID":
        return "Identifier — link predictions to customers; exclude from modeling (per AGENTS.md)"
    if col == "Churn":
        return "Target — binary class (Yes/No)"
    if col == "TotalCharges":
        return "Numeric feature candidate — currently object dtype; requires cleaning/coercion"
    if col == "SeniorCitizen":
        return "Binary feature — stored as int64 (0/1)"
    if col in {"tenure", "MonthlyCharges"}:
        return "Numeric feature"
    if df[col].nunique() == 2:
        return "Binary categorical feature"
    if df[col].nunique() == 3:
        return "Multi-class categorical feature (3 levels)"
    return "Feature — review during preprocessing"


# Missing values: include explicit NaN plus blank TotalCharges for inspection accuracy
missing_values = df.isnull().sum()
if "TotalCharges" in df.columns:
    blank_total_charges = df["TotalCharges"].astype(str).str.strip().eq("").sum()
    missing_values["TotalCharges"] = missing_values["TotalCharges"] + blank_total_charges

data_dictionary = pd.DataFrame({
    "Column": df.columns,
    "Data Type": df.dtypes.astype(str).values,
    "Unique Values": df.nunique().values,
    "Missing Values": [missing_values[c] for c in df.columns],
    "Initial Role / Note": [initial_role_note(c) for c in df.columns],
})

data_dictionary

## Initial Data Quality Findings

Summary based only on inspection of `WA_Fn-UseC_-Telco-Customer-Churn.csv`:

- **Shape:** 7,043 rows × 21 columns.
- **Target (`Churn`):** 5,174 No (73.46%), 1,869 Yes (26.54%) — moderate class imbalance.
- **Missing values (pandas `isnull`):** No explicit NaN values in any column.
- **Duplicate rows:** 0.
- **`customerID`:** 7,043 unique values — one ID per row; suitable as a linkage key only, not a predictive feature.
- **`TotalCharges`:** Stored as `object`, not numeric. Eleven rows contain blank/whitespace strings; all 11 have `tenure = 0`. These rows coalesce to NaN when converted with `pd.to_numeric(..., errors="coerce")`. Cleaning/coercion will be required before modeling.
- **`tenure`:** Integer months; 11 customers have tenure 0 (same rows as blank `TotalCharges`).
- **`MonthlyCharges`:** Float64 with no missing values; range approximately 18.25–118.75.
- **`SeniorCitizen`:** Already encoded as 0/1 integer (not Yes/No strings).
- **Sentinel categories:** Several service columns include `"No internet service"` or `"No phone service"` levels tied to product absence — encoding strategy needed later, not cleaning now.
- **No train/test split, imputation, or modeling performed in this notebook.**